In [10]:
import re

# ---------------- NAME CLEANER ----------------

def clean_name(first, last):
    issues = []

    name = f"{first} {last}"

    # remove bracket content
    name = re.sub(r"\(.*?\)", "", name)

    # remove titles
    titles = [
        "mr", "mrs", "ms", "dr", "prof", "sir", "madam",
        "er", "engr", "eng", "md", "mohd", "mohammed"
    ]

    words = name.split()
    words = [w for w in words if w.lower().strip(".") not in titles]

    name = " ".join(words)

    # remove digits
    name = re.sub(r"\d+", "", name)

    # remove special characters
    name = re.sub(r"[^a-zA-Z\s]", "", name)

    # remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    # validation
    if len(name) < 3:
        issues.append("Name too short or invalid")

    if len(name.split()) == 1:
        issues.append("Only one name detected (ATS prefers first and last name)")

    # proper case
    name = name.title()
    ats_name = name.upper()

    initials = "".join([w[0] for w in name.split()])

    return ats_name, initials, issues


# ---------------- PHONE CLEANER ----------------

def clean_phone(phone, country_code="+91"):
    digits = re.sub(r"\D", "", phone)

    if digits.startswith("0") and len(digits) > 10:
        digits = digits[1:]

    if len(digits) == 10:
        return f"{country_code} {digits[:5]} {digits[5:]}"
    elif len(digits) > 10 and digits.startswith("91"):
        digits = digits[2:]
        return f"{country_code} {digits[:5]} {digits[5:]}"
    else:
        return None


# ---------------- EMAIL ----------------

def validate_email(email):
    original = email
    issues = []

    # remove spaces and lowercase
    email = email.strip().lower().replace(" ", "")

    # common domain typos
    domain_fixes = {
        "gamil.com": "gmail.com",
        "gmial.com": "gmail.com",
        "gmail.co": "gmail.com",
        "gmail,com": "gmail.com",
        "yaho.com": "yahoo.com",
        "outlok.com": "outlook.com",
        "hotnail.com": "hotmail.com"
    }

    # missing @ symbol
    if "@" not in email:
        issues.append("Missing '@' symbol")

        # attempt auto-fix for gmail
        if "gmail" in email:
            name = email.replace("gmail.com", "").replace("gmail", "")
            email = name + "@gmail.com"
        else:
            return original, False, issues

    # split username/domain
    try:
        username, domain = email.split("@", 1)
    except ValueError:
        return original, False, ["Invalid email structure"]

    # fix known domain mistakes
    if domain in domain_fixes:
        issues.append(f"Corrected domain '{domain}' → '{domain_fixes[domain]}'")
        domain = domain_fixes[domain]

    # add .com if missing
    if "." not in domain:
        issues.append("Added '.com' to domain")
        domain += ".com"

    corrected_email = f"{username}@{domain}"

    # final validation
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    valid = bool(re.match(pattern, corrected_email))

    return corrected_email, valid, issues


def suggest_professional_email(first, last):
    return f"{first.lower()}.{last.lower()}@gmail.com"


# ---------------- URL NORMALIZER ----------------

def normalize_url(url, base):
    if not url or url.strip() == "":
        return None

    url = url.strip().lower()

    # remove protocol
    url = url.replace("https://", "").replace("http://", "")

    # remove www
    if url.startswith("www."):
        url = url[4:]

    # remove duplicate slashes
    url = url.replace("//", "/")

    # remove trailing slash
    url = url.rstrip("/")

    # already full link
    if base in url:
        return "https://" + url

    # username only
    return f"https://{base}/{url}"



def clean_linkedin(linkedin):
    return normalize_url(linkedin, "linkedin.com/in")


def clean_github(github):
    return normalize_url(github, "github.com")


# ---------------- CITY CLEANER ----------------

def clean_city(city):
    city_map = {
        "hyd": "Hyderabad",
        "blr": "Bengaluru",
        "mum": "Mumbai",
        "del": "Delhi",
        "sec": "Secunderabad"
    }

    city = city.strip().lower()
    return city_map.get(city, city.title())


# ---------------- MAIN FUNCTION ----------------

def format_personal_details(first, last, phone, email, linkedin, github, city):

    issues = []

    # NAME
    full_name, initials, name_issues = clean_name(first, last)
    issues.extend(name_issues)

    # PHONE
    phone_clean = clean_phone(phone)
    if phone_clean is None:
        issues.append("Invalid phone number format")
        phone_clean = "N/A"

    # EMAIL
    # EMAIL
    email_clean, valid, email_issues = validate_email(email)
    issues.extend(email_issues)

    if not valid:
        issues.append("Email could not be corrected automatically")
    
    
    


    bad_words = ["killer", "gaming", "boy", "king", "pro"]
    if any(word in email_clean for word in bad_words):
        suggestion = suggest_professional_email(first, last)
        issues.append(f"Unprofessional email. Suggested: {suggestion}")

    # LINKS
    linkedin_clean = clean_linkedin(linkedin)
    if linkedin_clean is None:
        issues.append("LinkedIn missing")
        linkedin_clean = "N/A"

    github_clean = clean_github(github)
    if github_clean is None:
        issues.append("GitHub missing")
        github_clean = "N/A"

    # LOCATION
    city_clean = clean_city(city)
    location = f"{city_clean}, India"

    # SHORT LINKS FOR RESUME (ATS readable, less space)
    linkedin_short = linkedin_clean.replace("https://", "")
    github_short = github_clean.replace("https://", "")

    # ATS SAFE COMPACT HEADER
    header = f"""{full_name}
{phone_clean} | {email_clean} | {location}
LinkedIn: {linkedin_short}
GitHub: {github_short}
"""

    return header, issues, initials

In [9]:
print("=== Resume Header Generator ===\n")

first = input("First Name: ")
last = input("Last Name: ")
phone = input("Phone Number: ")
email = input("Email Address: ")
linkedin = input("LinkedIn (username or URL): ")
github = input("GitHub (username or URL): ")
city = input("City: ")

header, issues, initials = format_personal_details(
    first, last, phone, email, linkedin, github, city
)

print("\n========= GENERATED HEADER =========\n")
print(header)

if issues:
    print("\n⚠ Issues detected:")
    for i in issues:
        print("-", i)
else:
    print("\nNo issues detected. Ready to use!")

# Save file
with open("resume_header.txt", "w", encoding="utf-8") as f:
    f.write(header)

print("\nSaved as resume_header.txt")

=== Resume Header Generator ===


========= GENERATED HEADER =========

SOHEL MAHAMMAD
+91 96036 07597 | sohel.tensa@gmail.com | Hyderabad, India
LinkedIn: linkedin.com/in/md-sohel-8a5419187
GitHub: github.com/tensakuro


No issues detected. Ready to use!

Saved as resume_header.txt
